# 01 · Rounded square, bed as master
The bed turns continuously and its angle is always the polar angle of the point being laid, so the arm sits on one line out from
the axis and only slides in and out. The bed speed follows the outline so the bead speed is constant, and the extruder follows the
*measured* bed speed. Nothing stops, so there are no corners to hand over.

In order: settings → geometry (look at it) → plan (numbers, no motion) → connect → prime → the loop → finish → read the log. The tested reference for all of this is `src/cell/runs/square_polar.py`; the loop cell is the same code laid out so you can read and change it.

## Settings

In [ ]:
import sys, os, time, math, threading
ROOT = os.path.abspath("..")
sys.path[:0] = [ROOT, os.path.join(ROOT, "src"), os.path.join(ROOT, "ui")]   # not needed after: pip install -e .

import cell.runs.square_polar as sp
from cell.runs.cylinder import Cell, PARK, SAFE_Z, WATCHDOG_S, SIDE

SETTINGS = dict(
    T=0,            # 0 right, 1 left, 2 both
    SIDE=80,        # outside side length, mm (bead centreline)
    RF=10,          # fillet radius, mm
    L_H=0.6,        # bead height, mm
    W=1.0,          # bead width, mm (1.0 mm nozzle)
    FLOW_K=1.3,     # measured / planned plastic from a test print
    V=8,            # bead speed along the outline, mm/s
    LOOPS=10,       # total loops (first and last are flat)
    BED_ACC=500,    # ask the Aerotech for this ramp, deg/s2 (0 = leave)
)
SIM = True          # simulators first; False for the cell

g, _ = sp.parse([f"{k}={v}" for k, v in SETTINGS.items()])
arms = g["arms"]
lead = arms[min(arms)]
print("per-arm:", {SIDE[t]: {k: round(a[k], 3) for k in ("S", "R", "D", "K", "Z0")} for t, a in arms.items()})

## Geometry
`radius(a, θ)` is the outline's distance from the axis at polar angle θ (θ = 0 faces the arm). `ds_dtheta` is how many millimetres of
outline pass per radian of bed turn; the bed speed is `V / ds_dtheta`.

In [ ]:
import matplotlib.pyplot as plt
th = [i * 0.5 for i in range(0, 721)]
r = [sp.radius(lead, t) for t in th]
w = [g["V"] / sp.ds_dtheta(lead, t) for t in th]
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot([ri * math.cos(math.radians(t)) for ri, t in zip(r, th)], [ri * math.sin(math.radians(t)) for ri, t in zip(r, th)])
ax[0].set_aspect("equal"); ax[0].set_title("outline on the disc (mm)"); ax[0].plot(0, 0, "k+")
ax[1].plot(th, r); ax[1].set_title("arm radius r(θ), mm"); ax[1].set_xlabel("bed angle, deg")
ax[2].plot(th, w); ax[2].set_title("bed speed ω(θ), rad/s"); ax[2].set_xlabel("bed angle, deg")
plt.tight_layout(); plt.show()
print(f"r from {min(r):.1f} to {max(r):.1f} mm;  bed {min(w):.3f} to {max(w):.3f} rad/s")

## Plan (numbers only, nothing moves)

In [ ]:
k = sp.plan(g)
sp.show(g, k)

## Connect

In [ ]:
cell = Cell(g, SIM)
per = k["per"]
rate = {tl: 0.0 for tl in arms}                       # filament mm/s wanted per tool; the thread below sends it

def gcode(s): return cell.gcode(s)

def extruder_thread():                                 # sends EXTRUDE_ON when the wanted rate changes, keeps the watchdog fed
    sent, sent_at = {tl: None for tl in arms}, {tl: 0.0 for tl in arms}
    period = 1.0 / g["RATE_HZ"]
    while not cell.stop_keep.wait(period / 2):
        now = time.monotonic()
        for tl in arms:
            rr = rate[tl]
            if rr < g["RATE_MIN"]:
                if sent[tl] is not None:
                    gcode(f"EXTRUDE_OFF T={tl}"); sent[tl] = None
                continue
            due = (sent[tl] is None or abs(rr - sent[tl]) > sp.RATE_TOL * sent[tl]) and now - sent_at[tl] >= period
            if due or (sent[tl] is not None and now - sent_at[tl] >= WATCHDOG_S / 2.5):
                gcode(f"EXTRUDE_ON T={tl} RATE={rr:.3f} TIMEOUT={WATCHDOG_S}"); sent[tl], sent_at[tl] = rr, now

def pt(a, r):                                           # radius along the arm's print direction -> X, Y in its frame
    ang = math.radians(a["ANGLE"]); return (a["CX"] + r * math.cos(ang), a["CY"] + r * math.sin(ang))

def move(tl, r, z, speed, wait=False, blend=2.0):
    if z < arms[tl]["Z0"] - 1e-6: raise RuntimeError(f"{SIDE[tl]} Z {z:.2f} below Z0")
    ro, pi_, yw = cell.orient[tl]; x, y = pt(arms[tl], r)
    cell.arms[tl].arm.set_position(x, y, z, ro, pi_, yw, speed=speed, wait=wait, radius=blend)

gcode("EXTRUDE_OFF")
for tl in (0, 1): gcode(f"EXTRUDE_SCALE T={tl} PERCENT=100")
cell.tt.stop_rotation()
if g["BED_ACC"] > 0: sp.set_bed_ramp(cell.tt, g["BED_ACC"])
if not SIM:
    for tl in arms: assert cell.hot_end_temp(tl) >= g["MIN_TEMP"], f"{SIDE[tl]} hot end cold"
print("ready")

## Approach and prime
Arms go to park, up to the safe height, out over the prime point (15 mm outside the outline), down. Then the bed turns at half the
side-centre speed while the extruder runs at half rate and the nozzle slides in to the outline and the first layer height.

In [ ]:
for tl, a in arms.items():
    x, y = pt(a, per[tl]["r_min"] + g["PRIME_MM"])
    cell.move(tl, *PARK, speed=k["arm_speed"], wait=True)
    cell.move(tl, PARK[0], PARK[1], SAFE_Z, speed=k["arm_speed"], wait=True)
    cell.move(tl, x, y, per[tl]["prime_z"] + 20, speed=k["arm_speed"], wait=True)
    cell.move(tl, x, y, per[tl]["prime_z"], speed=20, wait=True)

a0 = cell.tt.get_angle()                                      # θ = 0: a side centre faces the arm
w0 = g["V"] / sp.ds_dtheta(lead, 0.0)
cell.tt.rotate_velocity(math.degrees(w0) / 2)
for tl in arms: rate[tl] = per[tl]["rate"] / 2
threading.Thread(target=extruder_thread, daemon=True).start()
t0 = time.monotonic()
while time.monotonic() - t0 < g["PRIME_S"]:
    f = (time.monotonic() - t0) / g["PRIME_S"]
    th = cell.tt.get_angle() - a0
    for tl, a in arms.items():
        move(tl, sp.radius(a, th) + g["PRIME_MM"] * (1 - f), per[tl]["prime_z"] + (per[tl]["z_first"] - per[tl]["prime_z"]) * f, 10)
    time.sleep(sp.TICK)
print("primed; bed at %.1f deg" % (cell.tt.get_angle() - a0))

## The loop
Every 50 ms: read the bed angle θ → measured bed speed → **bed** gets the profile speed a little ahead (`LEAD_S`) → **extruder**
gets `K × ω_measured × ds/dθ` → **arm** gets `r(θ + lead)` and the Z for this point of the spiral. One loop = 360° of bed.

In [ ]:
th_end = 360.0 * k["total_loops"]
prev_th, prev_t, w_filt = cell.tt.get_angle() - a0, time.monotonic(), w0
last = {tl: None for tl in arms}
log = []
while True:
    now = time.monotonic()
    th = cell.tt.get_angle() - a0
    if th >= th_end: break
    w_raw = math.radians((th - prev_th) / max(now - prev_t, 1e-3))
    w_filt = 0.5 * w_filt + 0.5 * max(0.0, w_raw)
    th_ahead = th + math.degrees(w_filt) * g["LEAD_S"]
    cell.tt.rotate_velocity(math.degrees(min(g["V"] / sp.ds_dtheta(lead, th_ahead), g["MAX_VB"])))
    for tl, a in arms.items():
        rate[tl] = a["K"] * w_filt * sp.ds_dtheta(a, th)
        r, z = sp.radius(a, th_ahead), per[tl]["z_first"] + sp.height(g, th_ahead / 360.0)
        if last[tl] is None or abs(r - last[tl][0]) >= 0.05 or abs(z - last[tl][1]) >= 0.02:
            move(tl, r, z, max(10.0, 3 * k["rdot"])); last[tl] = (r, z)
    log.append((now - t0, th, w_filt, *[rate[tl] for tl in arms], *[cell.pose(tl)[2] for tl in arms]))
    prev_th, prev_t = th, now
    time.sleep(sp.TICK)
print("done: %.2f loops" % (th / 360.0))

## Finish: extruders off, bed stop, lift, park, square up
The bed ends wherever the loop crossed its end. The last step turns it to the nearest angle at which a face centre points at the arm (every 90° from the start angle), so the ironing notebook can begin square to a side. `FACE_ANGLE` is that angle, in case the next notebook needs it.

In [ ]:
cell.stop_keep.set()
gcode("EXTRUDE_OFF"); cell.tt.stop_rotation()
for tl, a in arms.items():
    x, y = pt(a, sp.radius(a, th)); z = per[tl]["z_first"] + sp.height(g, th / 360.0)
    cell.move(tl, x, y, z + 10, speed=20, wait=True)
for tl in arms: cell.move(tl, *PARK, speed=k["arm_speed"], wait=True)
FACE_ANGLE = a0 + 90.0 * round((cell.tt.get_angle() - a0) / 90.0)     # nearest angle with a face centre facing the arm
sp.square_up(cell.tt, FACE_ANGLE)
print(f"finished; bed at {cell.tt.get_angle():.2f} deg, a face centre faces the arm (FACE_ANGLE = {FACE_ANGLE:.2f})")

## Read the run

In [ ]:
import pandas as pd
cols = ["t_s", "bed_deg", "bed_rad_s"] + [f"rate_{SIDE[t]}" for t in arms] + [f"z_{SIDE[t]}" for t in arms]
df = pd.DataFrame(log, columns=cols)
ax = df.plot(x="bed_deg", y=["bed_rad_s"] + [f"rate_{SIDE[t]}" for t in arms], subplots=True, figsize=(12, 6), grid=True)
plt.show()
df.describe().T